# Layer Norm, Residuals and Feed-Forward Sublayers

তিনটি demo:

1. Residual connection কীভাবে খুব গভীর stack-এর মধ্য দিয়ে gradient-কে জীবিত রাখে, PyTorch autograd দিয়ে সরাসরি মাপা ("+x" skip connection সহ বনাম ছাড়া তুলনা)।
2. হাতে লেখা একটি LayerNorm, `nn.LayerNorm`-এর সাথে সংখ্যাগতভাবে যাচাই করা।
3. Parameter count: attention বনাম feed-forward sublayer, README-এর দাবি "একটি layer-এর parameter-এর ~2/3 FFN-এ থাকে" নিশ্চিত করা।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Residual connection depth জুড়ে gradient-কে জীবিত রাখে

একটি ছোট `SimpleBlock` (Linear -> Tanh -> Linear, ছোট init) বহুবার stack করে একটি `DeepStack` বানানো হয়। বিভিন্ন depth-এ residual (`x = x + block(x)`) সহ ও ছাড়া (`x = block(x)`) input-এ পৌঁছানো gradient norm মাপা হয় — residual ছাড়া depth বাড়লে gradient শূন্যের দিকে চলে যায়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Residual connection depth জুড়ে gradient-কে জীবিত রাখে
# ---------------------------------------------------------------------------

class SimpleBlock(nn.Module):
    """একটি ন্যূনতম stand-in sublayer: Linear -> Tanh -> Linear, ছোট init।"""

    def __init__(self, d, seed_offset=0):
        super().__init__()
        torch.manual_seed(seed_offset)
        self.fc1 = nn.Linear(d, d)
        self.fc2 = nn.Linear(d, d)
        with torch.no_grad():
            self.fc1.weight *= 0.5
            self.fc2.weight *= 0.5

    def forward(self, x):
        return self.fc2(torch.tanh(self.fc1(x)))


class DeepStack(nn.Module):
    def __init__(self, d, depth, use_residual):
        super().__init__()
        self.blocks = nn.ModuleList([SimpleBlock(d, seed_offset=i) for i in range(depth)])
        self.use_residual = use_residual

    def forward(self, x):
        for block in self.blocks:
            x = x + block(x) if self.use_residual else block(x)
        return x


def residual_gradient_demo():
    print("=" * 70)
    print("1. RESIDUAL CONNECTIONS AND GRADIENT FLOW THROUGH DEPTH")
    print("=" * 70)
    print("Measuring the gradient norm that reaches the INPUT of a deep stack,")
    print("with vs. without residual ('x = x + block(x)' vs. 'x = block(x)').\n")

    d = 16
    print(f"{'depth':>6}  {'grad norm, NO residual':>24}  {'grad norm, WITH residual':>26}")
    for depth in [5, 10, 20, 40, 80]:
        x_no_res = torch.randn(d, requires_grad=True)
        stack_no_res = DeepStack(d, depth, use_residual=False)
        out_no_res = stack_no_res(x_no_res).sum()
        out_no_res.backward()
        grad_norm_no_res = x_no_res.grad.norm().item()

        x_res = torch.randn(d, requires_grad=True)
        stack_res = DeepStack(d, depth, use_residual=True)
        out_res = stack_res(x_res).sum()
        out_res.backward()
        grad_norm_res = x_res.grad.norm().item()

        print(f"{depth:>6}  {grad_norm_no_res:>24.8f}  {grad_norm_res:>26.8f}")

    print("\n-> Without residual connections, the gradient reaching the input")
    print("   shrinks toward zero as depth grows -- the same vanishing-signal")
    print("   story as Phase 01's RNN gradient decay, just caused by stacking")
    print("   layers instead of stacking timesteps. With residual connections,")
    print("   the identity path (+x) guarantees a gradient of at least 1 flows")
    print("   straight back regardless of depth, keeping the total gradient")
    print("   from collapsing.")


residual_gradient_demo()

## 2. হাতে লেখা LayerNorm বনাম `nn.LayerNorm`

শুধু শেষ dimension-এর উপর (প্রতি token আলাদাভাবে, প্রতি batch নয়) normalize করে এমন একটি manual LayerNorm লেখা হয় এবং একই gamma/beta দিয়ে `torch.nn.LayerNorm`-এর output-এর সাথে তুলনা করা হয় — পার্থক্য floating-point স্তরে নগণ্য।

In [ ]:
# ---------------------------------------------------------------------------
# 2. হাতে লেখা LayerNorm বনাম nn.LayerNorm
# ---------------------------------------------------------------------------

def manual_layer_norm(x, gamma, beta, eps=1e-5):
    """x: (..., d)। কেবল শেষ dimension-এর উপর normalize করে (প্রতি-token, প্রতি-batch নয়)।"""
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_norm + beta


def layer_norm_demo():
    print("\n" + "=" * 70)
    print("2. HAND-ROLLED LayerNorm vs. torch.nn.LayerNorm")
    print("=" * 70)

    batch, T, d_model = 2, 4, 8
    x = torch.randn(batch, T, d_model) * 5 + 3   # ইচ্ছাকৃতভাবে mean-0/std-1 নয়

    torch_ln = nn.LayerNorm(d_model)
    with torch.no_grad():
        # ন্যায্য তুলনার জন্য উভয়ের ক্ষেত্রে একই (randomly initialized) gamma/beta ব্যবহার করুন।
        gamma, beta = torch_ln.weight.clone(), torch_ln.bias.clone()

    manual_output = manual_layer_norm(x, gamma, beta)
    torch_output = torch_ln(x)

    max_diff = (manual_output - torch_output).abs().max().item()
    print(f"Input stats:  mean={x.mean().item():.3f}  std={x.std().item():.3f}")
    print(f"Output stats (torch.nn.LayerNorm), per-token: mean~0, std~1 before "
          f"gamma/beta scaling")
    print(f"Max abs difference between manual and torch.nn.LayerNorm: {max_diff:.2e}")

    per_token_mean = torch_output.mean(dim=-1)
    print(f"\nPer-token means after LayerNorm: {per_token_mean[0].detach().numpy().round(3)}")
    print("(Exactly 0 here only because nn.LayerNorm's beta defaults to all-zeros and")
    print(" gamma to all-ones; with a trained/non-default beta the mean would shift to")
    print(" beta's own mean, since normalization guarantees mean 0 BEFORE gamma/beta.)")


layer_norm_demo()

## 3. Parameter count: attention বনাম feed-forward

GPT-2-small মাপে (`d_model=768`, `d_ff=4*d_model`) attention sublayer (`W_q, W_k, W_v, W_o`) ও feed-forward sublayer (`fc1, fc2`)-এর parameter গোনা হয় — FFN একটি layer-এর মোট parameter-এর প্রায় দুই-তৃতীয়াংশ ধারণ করে।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Parameter count: attention বনাম feed-forward
# ---------------------------------------------------------------------------

class MultiHeadAttentionParams(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


class FeedForwardParams(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)


def count_params(module):
    return sum(p.numel() for p in module.parameters())


def parameter_count_demo():
    print("\n" + "=" * 70)
    print("3. PARAMETER COUNTS: ATTENTION vs. FEED-FORWARD SUBLAYER")
    print("=" * 70)

    d_model = 768   # বাস্তবসম্মত তুলনার জন্য GPT-2-small মাপ
    d_ff = 4 * d_model

    attn = MultiHeadAttentionParams(d_model)
    ffn = FeedForwardParams(d_model, d_ff)

    attn_params = count_params(attn)
    ffn_params = count_params(ffn)
    total = attn_params + ffn_params

    print(f"d_model={d_model}, d_ff={d_ff} (the conventional 4x d_model)")
    print(f"  Attention sublayer params: {attn_params:,}")
    print(f"  Feed-forward sublayer params: {ffn_params:,}")
    print(f"  FFN's share of one layer's total: {ffn_params / total:.1%}")
    print("\n-> Matches the README's claim: the feed-forward sublayer holds")
    print("   roughly two-thirds of a Transformer layer's parameters, even")
    print("   though self-attention gets most of the conceptual spotlight.")


parameter_count_demo()

## সবগুলো demo একসাথে

`main()` উপরের সবগুলো demo এক সাথে ক্রমানুসারে চালায়। প্রতিটি demo ইতিমধ্যে নিজের cell-এ চলেছে, তাই দ্বিতীয়বার না চালানোর জন্য নিচে call-টি comment করা আছে।

In [ ]:
def main():
    residual_gradient_demo()
    layer_norm_demo()
    parameter_count_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন